# Project 2 - Recipe Nutrition Agent

#### Objective: Build an agent that uses three tools to analyse a dish and returns a validated, nested Pydantic object.

##### Test
```I want to make a mushroom risotto for one person. How many calories is it, and is it vegetarian?```

##### Expected Output
```
MealAnalysis(
dish='mushroom risotto',
ingredients=[Ingredient(name='arborio rice', grams=80.0), Ingredient(name='mushrooms', grams=150.0)
total_calories=610,
dietary_flags=['vegetarian', 'gluten_free'],
confidence='high',
)
```

#### Step 1: Prerequisite

In [45]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from data_p2 import RECIPES, NUTRITION, DIETARY, ALL_FLAGS
from pprint import pprint

# Load Environment Variables
load_dotenv(override=True)

True

#### Step 2: Declare Pydantic Objects

In [46]:
class Ingredient(BaseModel):
    name: str
    grams: float = Field(gt=0)

class MealAnalysis(BaseModel):
    dish: str
    ingredients: list[Ingredient]
    total_calories: int = Field(ge=0)
    dietary_flags: list[Literal['vegetarian', 'vegan', 'gluten_free', 'dairy_free']]
    confidence: Literal['low', 'medium', 'high']

#### Step 3: Create Tools 

In [47]:
# Tool - Return Ingredients With Grams
@tool
def find_recipe(dish: str, servings: int) -> list[Ingredient] | str:
    '''Return the ingredients for a dish with total grams for the given servings.
    Return an ERROR string if the dish is not known.'''
    if dish not in RECIPES:
        return f"ERROR: no recipe found for '{dish}'. Known dishes: {', '.join(RECIPES)}"
    return [Ingredient(name=k, grams=v * servings) for k, v in RECIPES[dish].items()]

# Tool To Get Nutrition
@tool
def get_nutrition(ingredient: str, grams: float) -> str:
    '''Return the calories (kcal) for the given grams of one ingredient.
    Returns an ERROR string if the ingredient has no nutrition data.'''
    if ingredient not in NUTRITION: 
        return f"ERROR: no nutrition data for '{ingredient}'."
    total_cal = (NUTRITION[ingredient]/100) * grams
    return f"{ingredient}: {total_cal} kcal"

# Tool To Check Dietary Flags
@tool
def check_dietary(ingredients: list[str]) -> str:
    '''Construct a list of dietary flags from the ingredients and sent back a list of intersecting dietary flags.'''
    if not ingredients:
        return 'No dietary flags applied. Empty ingredients'
    flags = set(DIETARY.get(ingredients[0], set()))
    for i in ingredients[1:]:
        flags.intersection_update(DIETARY.get(i, set()))
    return ', '.join(flags)

#### Step 4: Create Agent

In [48]:
SYSTEM_PROMPT = """Your task is to analyze the meal from user query. 
You are given a range of tools. Use find_recipe to find ingredients, get_nutrition to get the calories,
and check_dietary to check any dietary flags. You should not estimate the unknown data yourself.
If any of the tools returns an error, set confidence level to low and leave that specific ingredient or its calories out.
If the dish is not found, return an empty ingredients list, total_calories=0,
  dietary_flags=[], and confidence='low'.
"""

agent = create_agent(model = 'openai:gpt-4o-mini', 
             tools = [find_recipe, get_nutrition, check_dietary], 
             system_prompt = SYSTEM_PROMPT,
             response_format=MealAnalysis)

In [49]:
# Call Agent
response = agent.invoke({"messages": [
        HumanMessage(content="I want to make a mushroom risotto for one person. How many calories is it, and is it vegetarian?")]})
print(response['messages'][-1])

content='{"dish":"mushroom risotto","ingredients":[{"name":"arborio rice","grams":80},{"name":"mushrooms","grams":150},{"name":"onion","grams":50},{"name":"vegetable stock","grams":250},{"name":"parmesan","grams":20},{"name":"butter","grams":10},{"name":"olive oil","grams":10}],"total_calories":601,"dietary_flags":["gluten_free"],"confidence":"high"}' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 876, 'total_tokens': 981, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_402fb8a3d9', 'id': 'chatcmpl-EcTra8z5LUEOiWj7TP19FMYskl9vE', 'service_tier': 'default', 'finish_re

In [50]:
# Print Agent Response
print(response["structured_response"].model_dump_json(indent=2))

{
  "dish": "mushroom risotto",
  "ingredients": [
    {
      "name": "arborio rice",
      "grams": 80.0
    },
    {
      "name": "mushrooms",
      "grams": 150.0
    },
    {
      "name": "onion",
      "grams": 50.0
    },
    {
      "name": "vegetable stock",
      "grams": 250.0
    },
    {
      "name": "parmesan",
      "grams": 20.0
    },
    {
      "name": "butter",
      "grams": 10.0
    },
    {
      "name": "olive oil",
      "grams": 10.0
    }
  ],
  "total_calories": 601,
  "dietary_flags": [
    "gluten_free"
  ],
  "confidence": "high"
}


In [51]:
# Call Agent - Error Meal
response = agent.invoke({"messages": [
        HumanMessage(content="I want to make a chinese risotto for one person. How many calories is it, and is it vegetarian?")]})
print(response['messages'][-1])

content='{"dish":"chinese risotto","ingredients":[],"total_calories":0,"dietary_flags":[],"confidence":"low"}' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 505, 'total_tokens': 536, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_402fb8a3d9', 'id': 'chatcmpl-EcTsBQ8gOdrUGGyNgl9tHAdjNNPCe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a16d97-6bb1-747e-8495-e5d51cf12bde-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 505, 'output_tokens': 31, 'total_tokens': 536, 'input_token_details': {'audio': 0, 'cache_r

In [52]:
print(response["structured_response"].model_dump_json(indent=2))

{
  "dish": "chinese risotto",
  "ingredients": [],
  "total_calories": 0,
  "dietary_flags": [],
  "confidence": "low"
}


In [54]:
for m in response["messages"]:
    if isinstance(m, ToolMessage):
        print(m.name, "→", m.content)

find_recipe → ERROR: no recipe found for 'chinese risotto'. Known dishes: mushroom risotto, spaghetti carbonara, chickpea curry, truffle mushroom pasta
